# 2. Comparación emparejada de calidad
Ejecutar tras el notebook 01. Solo lee las trazas de test exportadas por los adaptadores y usa el evaluador común. Si difieren timestamps, etiquetas o manifiesto, falla antes de mostrar métricas.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from shared.contract import ROOT
from shared.evaluation.evaluate import evaluate_pair
assert Path.cwd().resolve() == ROOT
traces = ROOT / 'results' / 'traces'
report = evaluate_pair(traces / 'lightgbm_night_test.parquet', traces / 'lightgbm_night_model2_isolated_test.parquet', traces / 'lstm_test.parquet', traces / 'lstm_model2_isolated_test.parquet')
assert report['paired_test_rows'] == 31680
print('Objetivos emparejados:', report['paired_test_rows'])


In [ ]:
sections = [('Model 1 binario', 'binary_full_test'), ('Model 2 aislado', 'model2_isolated_true_anomalies'), ('Model 2 condicionado', 'model2_conditional_detected_true_anomalies'), ('Cascada 7 clases', 'cascade_seven_class_full_test')]
rows = []
for approach in ('lightgbm', 'lstm'):
    for label, key in sections:
        item = report[approach][key]
        rows.append({'enfoque': approach, 'evaluación': label, 'objetivos': item['rows'], 'macro_precision': item['per_class']['macro avg']['precision'], 'macro_recall': item['per_class']['macro avg']['recall'], 'macro_f1': item['per_class']['macro avg']['f1-score']})
display(pd.DataFrame(rows))
display(pd.DataFrame([{'enfoque': a, 'falsas_alarmas_model1': report[a]['false_alarms_model1'], 'anomalias_omitidas_model1': report[a]['missed_anomalies_model1']} for a in ('lightgbm', 'lstm')]))


In [ ]:
for approach in ('lightgbm', 'lstm'):
    print('\n', approach, report[approach]['model_version'])
    for label, key in sections:
        item = report[approach][key]
        print(label)
        display(pd.DataFrame(item['confusion_matrix'], index=item['labels'], columns=item['labels']))
        display(pd.DataFrame(item['per_class']).T.loc[list(item['labels']), ['precision', 'recall', 'f1-score', 'support']])


El informe completo queda en `results/metrics/paired_test.json`, ignorado por Git. No mezclar Model 2 aislado, condicionado y la cascada: usan poblaciones diferentes.
